## Day 1: 일반 LLM은 왜 회사 매뉴얼에서 답을 찾지 못하는가

일반 LLM은 학습 중에 익힌 패턴으로 다음 토큰을 예측할 뿐이다 -- 무언가를
조회하는 메커니즘이 없으므로, 한 번도 공개된 적 없는 사적 문서에 대한
질문에는 환각으로 지어낸 답이나 (운이 좋으면) 거부만 나온다. RAG는
답변 시점에 관련 문단을 모델에게 직접 건네줌으로써 이 문제를 해결한다 --
오픈북 시험이 기억에만 의존하는 것보다 나은 것과 같은 원리다.

이 셀은 같은 질문에 대해 근거 없는("닫힌 책") 답변과 근거 있는("열린
책") 답변을 대조한다 -- 둘 다 실제 LLM 호출을 대신하는 예시 코드다(이
환경에는 네트워크 접근이 없다), 하지만 그 형태는 Day 4의 실제 `answer()`
함수가 만들어내는 것과 정확히 같다.


In [ ]:
handbook_sections = [
    {"id": "sec-4.1", "text": "The office is closed on all federal holidays.", "page": 13},
    {"id": "sec-4.2", "text": "New hires accrue 12 vacation days in their first year.", "page": 14},
    {"id": "sec-4.3", "text": "Employees may take a scenic vacation to the mountains.", "page": 14},
    {"id": "sec-5.1", "text": "Paid time off requests must be submitted two weeks in advance.", "page": 18},
]

def ungrounded_answer(question: str) -> str:
    # 검색된 컨텍스트가 전혀 없는 일반 LLM 호출을 대신한다 -- 일반적인
    # 학습 데이터 패턴에만 의존할 수 있을 뿐, 이 회사의 실제 매뉴얼(애초에
    # 공개된 적이 없으므로 학습 데이터에도 없다)은 절대 알 수 없다.
    return "Typically, companies offer around 15 days of PTO for new employees."

def grounded_answer(question: str, retrieved: dict) -> str:
    # 검색된 스니펫만을 컨텍스트로 준 LLM 호출을 대신한다.
    return f"According to {retrieved['id']} (page {retrieved['page']}): \"{retrieved['text']}\""

question = "How many vacation days does a new hire get?"
retrieved = handbook_sections[1]  # 이 조회가 실제로 어떻게 자동화되는지는 Day 2/3에서 다룬다

print("Ungrounded:", ungrounded_answer(question))
print("Grounded:  ", grounded_answer(question, retrieved))


**RAG의 4단계 흐름** (`daily/Day1_llm-limits-and-rag-intro.ko.md`의 다이어그램
참고): 문서 -> 인덱싱(청킹 + 임베딩) -> 검색(top-k 유사도 검색) -> 생성
(검색된 텍스트에 국한된 답변). 이 노트북의 나머지 부분은 각 단계를 실제로
만든다: Day 2는 `embed()`와 유사도 수식을, Day 3은 저장/검색 계층을, Day
4는 청킹과 최종 `answer()` 함수를 만든다.


## Day 2: 텍스트를 비교 가능한 숫자로 바꾸기

"관련 있는 문단"을 자동으로 검색하려면 "이 질문이 이 문단과 의미상 얼마나
가까운가"를 나타내는 숫자가 필요하다. 그게 바로 임베딩과 유사도 점수다.
아래에서: 네트워크 호출도 학습된 모델도 없이 해싱만으로 만든 문자 3-그램
토이 임베딩, 손 계산과 numpy로 교차 검증한 코사인 유사도, 그리고 비교를
위한 scikit-learn 기반의 완전히 실행 가능한 실제 TF-IDF 임베딩을 다룬다.


In [ ]:
import hashlib, math

def toy_embed(text: str, dims: int = 64) -> list:
    # dims=64 -> 원본 텍스트 길이와 무관하게 모든 텍스트가 고정 길이
    # "지문" 벡터가 된다.
    vec = [0.0] * dims
    text = text.lower().replace(" ", "_")
    for i in range(len(text) - 2):
        trigram = text[i:i + 3]
        # 3글자 윈도우를 64개 버킷 중 하나로 해싱한다. 서로 다른 3-그램이
        # 같은 버킷으로 충돌할 수 있다 -- 고정 크기 대 정밀도 사이의
        # 의도된 트레이드오프이지, 버그가 아니다.
        bucket = int(hashlib.md5(trigram.encode()).hexdigest(), 16) % dims
        vec[bucket] += 1.0
    return vec  # -> list[float], len == dims

def cosine_similarity(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))     # a를 b에 투영(정규화 전)
    norm_a = math.sqrt(sum(x * x for x in a))    # a의 길이
    norm_b = math.sqrt(sum(y * y for y in b))    # b의 길이
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

query = "how much paid time off do I get"
query_vec = toy_embed(query)
print("query vector length (dims):", len(query_vec))
print("nonzero buckets:", sum(1 for x in query_vec if x != 0), "out of", len(query_vec))

passages = [s["text"] for s in handbook_sections]
ranked = sorted(
    ((cosine_similarity(query_vec, toy_embed(doc)), doc) for doc in passages),
    reverse=True,
)
for score, doc in ranked:
    print(f"{score:.4f}  {doc}")


**주목할 한계:** 실제로 질문에 답하는 문장 -- "New hires accrue 12 vacation
days..." -- 이 네 개 중 **가장 마지막** 순위다. "The office is closed on
all federal holidays"가 1위인 이유는 순전히 실제 정답보다 쿼리와 3글자
부분 문자열을 더 많이 공유하기 때문이다. 토이 임베딩은 *철자*를 매칭할
뿐 *의미*를 매칭하지 못한다 -- 실제로 학습된 임베딩 모델이라면 "paid
time off"와 "vacation days"가 단어를 하나도 공유하지 않아도 가까이
배치할 것이다. 같은 맥락에서 쓰인다는 걸 학습했기 때문이다.


In [ ]:
import numpy as np

# 코사인 유사도에 대한 손으로 검증 가능한 기하학적 직관 -- 64차원 해싱
# 벡터 대신 단순한 2차원 벡터를 사용한다.
def cos_np(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

a = np.array([1.0, 0.0])   # 동쪽을 가리킨다
b = np.array([1.0, 1.0])   # 북동쪽(45도)을 가리킨다 -- a와 45도 차이
c = np.array([0.0, 1.0])   # 북쪽을 가리킨다 -- a와 90도 차이(직교)
d = np.array([5.0, 0.0])   # 동쪽을 가리킨다 -- a와 같은 방향, 길이만 5배

print("cos(a, b) =", cos_np(a, b), " expected cos(45 deg) =", math.cos(math.radians(45)))
print("cos(a, c) =", cos_np(a, c), " expected 0.0 (perpendicular = unrelated)")
print("cos(a, a) =", cos_np(a, a), " expected 1.0 (identical direction)")
print("cos(a, d) =", cos_np(a, d), " expected 1.0 (length dropped out -- direction is all that matters)")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

# TF-IDF: 문자 해싱보다 한 단계 발전한, 실제로 실행 가능한 방법 -- 컬렉션
# 전체에서 각 단어가 얼마나 특징적인지로 가중치를 매긴 단어 어휘 기반
# 벡터. 여전히 학습된 의미 모델은 아니지만, 임의의 3글자 조각을 매칭하지도
# 않는다.
vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(passages)     # shape: (문서 4개, V개 단어), sparse
query_vec_tfidf = vectorizer.transform([query])      # shape: (1, V개 단어), sparse

print("vocabulary size (V):", len(vectorizer.vocabulary_))
print("doc_matrix shape:", doc_matrix.shape)

sims = sk_cosine(query_vec_tfidf, doc_matrix)[0]     # shape: (4,) 문서마다 점수 하나
for score, doc in sorted(zip(sims, passages), reverse=True):
    print(f"{score:.4f}  {doc}")


TF-IDF는 1위는 정확히 맞춘다("paid time off requests"는 쿼리와 단어를
그대로 공유한다). 하지만 실제 의미상 정답인 "12 vacation days"는 여전히
정확히 0.0점이다 -- "paid time off"와 공유하는 단어가 하나도 없기 때문이다.
TF-IDF는 *철자* 문제는 고치지만 *동의어* 문제는 고치지 못한다. 이 간극을
메우는 건 오직 학습된 임베딩 모델뿐이다.


## Day 3: 임베딩을 대규모로 저장하기 -- 벡터 데이터베이스

쿼리를 저장된 벡터 하나하나와 비교하는 파이썬 반복문은 문단 4개에는
괜찮지만, 문서 청크 2만 개 이상에는 그렇지 않다: 쿼리당 O(n)이기
때문이다. 아래에서: 실제로 동작하는 인메모리 벡터 스토어
(create/add/query, 독자적인 인덱스 대신 numpy 행렬 하나로 뒷받침됨)와,
동일한 계산을 하는 순수 파이썬 반복문과의 실측 속도 비교를 다룬다.


In [ ]:
class InMemoryVectorStore:
    """
    실제 벡터 데이터베이스 collection의 대역: create/add/query 형태는
    동일하지만, 인덱스는 그냥 numpy 행렬이고, 검색은 ANN 인덱스 탐색이
    아니라 행렬-벡터 곱셈 한 번이다.
    """
    def __init__(self, dims: int):
        self.dims = dims
        self.ids, self.documents, self.metadatas = [], [], []
        self.embeddings = np.zeros((0, dims), dtype=np.float64)  # shape: (n_items, dims)

    def add(self, ids, documents, embeddings, metadatas):
        assert len(ids) == len(documents) == len(embeddings) == len(metadatas)
        self.ids.extend(ids)
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        new_block = np.array(embeddings, dtype=np.float64)          # shape: (n_new, dims)
        self.embeddings = np.vstack([self.embeddings, new_block])    # shape: (n_total, dims)

    def query(self, query_embedding, n_results: int = 3):
        if not self.ids:
            return []
        q = np.asarray(query_embedding, dtype=np.float64)  # shape: (dims,)
        # 행렬-벡터 곱셈 한 번으로 저장된 벡터 전체를 동시에 채점한다 --
        # 이게 바로 위 cosine_similarity 반복문을 벡터화한 것이다.
        with np.errstate(all="ignore"):  # 일부 플랫폼의 무해한 BLAS 경고 억제; 아래에서 NaN/inf 없음을 확인함
            dots = self.embeddings @ q                                        # shape: (n_items,)
            norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q)  # shape: (n_items,)
            sims = np.divide(dots, norms, out=np.zeros_like(dots), where=norms != 0)  # shape: (n_items,)
        top_idx = np.argsort(-sims)[:n_results]   # shape: (n_results,), 점수 높은 순
        return [
            {"id": self.ids[i], "document": self.documents[i],
             "metadata": self.metadatas[i], "score": float(sims[i])}
            for i in top_idx
        ]

collection = InMemoryVectorStore(dims=64)
collection.add(
    ids=[s["id"] for s in handbook_sections],
    documents=[s["text"] for s in handbook_sections],
    embeddings=[toy_embed(s["text"]) for s in handbook_sections],
    metadatas=[{"source": "handbook.pdf", "page": s["page"]} for s in handbook_sections],
)
print("embeddings matrix shape:", collection.embeddings.shape)

results = collection.query(toy_embed("how much PTO do new hires get"), n_results=3)
for r in results:
    print(f"{r['score']:.4f}  [{r['id']}] {r['document']}  {r['metadata']}")


`sec-4.2`("12 vacation days"라는 실제 정답)가 반환된 세 개 중 또다시
가장 마지막이라는 점을 눈여겨보자 -- 벡터 스토어의 `add()`/`query()`
메커니즘은 정확히 맞게 동작하고 있고, 다만 Day 2의 약한 토이 임베딩을
충실하게 검색하고 있을 뿐이다. **벡터 데이터베이스는 나쁜 임베딩을
빠르게 검색 가능하게 만들 뿐, 좋게 만들어주지는 않는다** -- 검색
품질은 저장 단계가 아니라 임베딩 단계에서 승부가 갈린다.


In [ ]:
import time

# 좀 더 현실적인 규모에서 벡터화가 만드는 실제 속도 향상을 측정한다:
# 무작위 64차원 벡터 20,000개를 위 스토어로 비교한 것과, 동일한
# 코사인 유사도 계산을 한 번에 하나씩 처리하는 순수 파이썬 반복문으로
# 비교한 것을 비교한다.
rng = np.random.default_rng(0)
n = 20_000
big_store = InMemoryVectorStore(dims=64)
big_store.add(
    ids=[f"doc-{i}" for i in range(n)],
    documents=["synthetic"] * n,
    embeddings=rng.random((n, 64)),   # shape: (20000, 64)
    metadatas=[{} for _ in range(n)],
)
q = rng.random(64)

t0 = time.perf_counter()
_ = big_store.query(q, n_results=5)
t1 = time.perf_counter()
vectorized_ms = (t1 - t0) * 1000
print(f"vectorized numpy query, n={n}: {vectorized_ms:.2f} ms")

def cosine_similarity_plain(u, v):
    # 의도적으로 순수 파이썬 수학 연산을 쓴다(호출마다 numpy를 쓰지
    # 않음) -- 이게 공정한 비교다: 동일한 산술 연산을 파이썬 수준에서
    # 하나씩 반복하는 것 vs. numpy가 한 번의 벡터화된 호출로 처리하는 것.
    dot = sum(x * y for x, y in zip(u, v))
    norm_u = math.sqrt(sum(x * x for x in u))
    norm_v = math.sqrt(sum(x * x for x in v))
    return dot / (norm_u * norm_v) if norm_u and norm_v else 0.0

t0 = time.perf_counter()
scored = sorted(
    ((cosine_similarity_plain(q, big_store.embeddings[i]), i) for i in range(n)),
    reverse=True,
)[:5]
t1 = time.perf_counter()
loop_ms = (t1 - t0) * 1000
print(f"pure python loop query, n={n}: {loop_ms:.2f} ms")
print(f"speedup: {loop_ms / vectorized_ms:.1f}x  (timing is machine-dependent -- order of magnitude is the point)")


**거리 vs. 유사도:** 대부분의 벡터 데이터베이스는 *거리*를
반환한다(작을수록 관련 있음) -- 흔히 `cosine_distance = 1 -
cosine_similarity`로 정의된다 -- 이는 위의 유사도 점수(클수록 관련
있음)와 반대 방향이다. 어떤 방향이든 정렬하기 전에 사용 중인
라이브러리가 어떤 규약을 쓰는지 항상 확인하자. 방향을 반대로 정렬하면
아무런 에러 없이 그럴듯해 보이지만 순서가 뒤바뀐 결과가 나온다.

프로덕션 규모에서 팀들은 보통 관리형 벡터 데이터베이스 서비스, 직접
운영하는 벡터 검색 엔진, 또는 이미 운영 중인 데이터베이스에 얹는 벡터
확장 기능을 사용한다 -- 위의 create/add/query 형태는 어떤 것을 쓰든
대체로 동일하게 유지된다.


## Day 4: 긴 문서를 청킹하고 검색된 컨텍스트만으로 답하기

전체 매뉴얼은 (질문마다 대부분 무관하기도 하고) 매 요청마다 보내기엔
너무 길다. overlap을 두고 청킹한 뒤, 특정 질문에 관련된 청크만 TF-IDF(Day
2) + 벡터 스토어(Day 3)로 검색하고, 그 청크만을 근거로 엄격하게
답한다 -- 관련된 게 없으면 정직하게 거부한다.


In [ ]:
def chunk_text(text: str, chunk_words: int = 40, overlap_words: int = 8) -> list:
    # 단어 수는 원시 문자 수보다, 문장 스타일이 다른 문서들 사이에서
    # 청크가 담은 "의미의 양"을 더 일관되게 나타낸다.
    words = text.split()                    # -> list[str]
    chunks, start = [], 0
    step = chunk_words - overlap_words        # 반복마다 창이 전진하는 폭
    while start < len(words):
        end = start + chunk_words
        chunks.append(" ".join(words[start:end]))   # -> str, chunk_words 이하 단어
        if end >= len(words):
            break
        start += step
    return chunks  # -> list[str]

handbook = """
Section 4.1: The office is closed on all federal holidays including New Year's Day,
Independence Day, and Thanksgiving. Employees are not required to use PTO for these days.
Section 4.2: New hires accrue 12 vacation days in their first year of employment, credited
monthly at a rate of one day per month. After three years of service the accrual rate
increases to 18 days per year.
Section 4.3: Paid time off requests must be submitted through the HR portal at least two
weeks in advance for any absence longer than two consecutive days. Same-day sick leave
does not require advance notice.
Section 5.1: Employee laptops are replaced every three years or upon failure, whichever
comes first. Submit a replacement request through the IT ticketing system.
"""

chunks = chunk_text(handbook, chunk_words=40, overlap_words=8)
print(f"chunked handbook ({len(handbook.split())} words) into {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {len(c.split())} words")


In [ ]:
# Day 2와 같은 TF-IDF 방식을 사용한, 청크에 대한 실제 종단 간(end-to-end) 검색.
chunk_vectorizer = TfidfVectorizer()
chunk_matrix = chunk_vectorizer.fit_transform(chunks)   # shape: (n_chunks, V)
print("chunk_matrix shape:", chunk_matrix.shape)

def search(query: str, top_k: int = 2):
    q_vec = chunk_vectorizer.transform([query])                    # shape: (1, V)
    sims = sk_cosine(q_vec, chunk_matrix)[0]                        # shape: (n_chunks,)
    ranked_idx = np.argsort(-sims)[:top_k]                          # shape: (top_k,)
    return [
        {"text": chunks[i], "score": float(sims[i]), "source": f"handbook.pdf#chunk{i}"}
        for i in ranked_idx
    ]

for h in search("how many vacation days do new hires get?"):
    print(f"{h['score']:.3f}  {h['source']}  {h['text'][:70]}...")


In [ ]:
import re

def call_llm(prompt: str) -> str:
    # 실제 LLM 호출을 대신하며, 주어진 컨텍스트에만 국한된다고 가정한다
    # (이 환경에는 네트워크 접근이 없다).
    return "New hires accrue 12 vacation days in their first year, credited monthly."

def answer(question: str, top_k: int = 2, min_relevance: float = 0.05) -> dict:
    hits = search(question, top_k=top_k)                     # Day 2/3의 검색
    relevant = [h for h in hits if h["score"] >= min_relevance]

    if not relevant:
        return {"answer": "I don't know -- nothing relevant was found in the documents.", "sources": []}

    context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in relevant)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    reply = call_llm(prompt)
    return {"answer": reply, "sources": [h["source"] for h in relevant]}

print("=== case 1: answerable question ===")
r1 = answer("how many vacation days do new hires get?")
print(r1)

# 저렴한 기계적 근거 검증: 답변이 인용한 모든 숫자는 실제로 검색된
# 컨텍스트 어딘가에 등장해야 한다(모델이 진짜 같은 출처를 인용하면서도
# 다른 숫자를 지어내는 걸 잡아낸다).
context_used = " ".join(h["text"] for h in search("how many vacation days do new hires get?"))
cited_numbers = re.findall(r"\d+", r1["answer"])
print("cited numbers:", cited_numbers, "-> all present in retrieved context:", all(n in context_used for n in cited_numbers))


In [ ]:
print("=== case 2: out-of-scope question (real pitfall) ===")
r2 = answer("what is the company's parental leave policy?")
print(r2)
print()
print("매뉴얼은 육아휴직을 전혀 언급하지 않지만, 검색된 두 청크 점수 모두")
print("0.05 임계값을 넘는다 -- 그래서 정직한 '모른다' 대신, 출처까지 달린")
print("그럴듯해 보이는 답변이 반환된다.")

print()
print("=== 이유: 불용어를 제거한 뒤에도 남는, 단어 하나짜리 어휘 중복 ===")
strict_vectorizer = TfidfVectorizer(stop_words="english")
strict_matrix = strict_vectorizer.fit_transform(chunks)
q_vec = strict_vectorizer.transform(["what is the company's parental leave policy?"])
sims = sk_cosine(q_vec, strict_matrix)[0]
for i, s in enumerate(sims):
    print(f"chunk {i}: {s:.3f}")
print()
print("청크 2만 0이 아닌 점수를 받는다 -- 쿼리와 'leave'라는 단어를 공유하기")
print("때문이다 ('same-day SICK LEAVE' vs. 'parental LEAVE policy') -- 같은")
print("단어지만 완전히 다른 주제다. 이건 Day 2의 철자-vs-의미 간극이 가장")
print("위험한 지점에서 다시 나타난 것이다: 정직한 거부를 자신 있게 인용된")
print("오답으로 뒤바꿀 수 있다.")


**이번 주의 정리:** RAG 시스템은 딱 그것이 답하기를 거부하는 정직함만큼만
믿을 수 있다. 여기 있는 모든 단계는 필요했다 -- "관련 있음"을 숫자로
만들기 위한 임베딩(Day 2), 그 검색을 대규모에서도 빠르게 만들기 위한
벡터 스토어(Day 3), 경계에서 사실을 잃지 않으면서 긴 문서를 검색 가능하게
만들기 위한 overlap 포함 청킹(Day 4) -- 하지만 위의 함정은 그중 어떤
것도 그 자체로 충분하지 않다는 걸 보여준다: 관련성 임계값과 그 밑바탕의
임베딩 모델 둘 다 실제 범위 밖 질문에 대해 튜닝되고 테스트되어야 하며,
그렇지 않으면 정직한 "모른다"가 가장 필요한 바로 그 지점에서 자신 있는
오답이 새어 나간다.
